In [1]:
import numpy as np
import mujoco
import mujoco.viewer                                                                                                                                                                                    
from mujoco_playground._src.manipulation.tesollo_hand import pinch
import xml
from env_viz import load_model
import pathlib
import ipynbname

In [2]:

_ROOT = ipynbname.path().parent.parent
xml_path = str(_ROOT / "mujoco_playground/_src/manipulation/tesollo_hand/xmls/scene_mjx_cube_pinch.xml")
spec = mujoco.MjSpec.from_file(xml_path)

In [7]:
spec.worldbody.frames
frame = spec.worldbody.frames[0]
print(frame.quat)

[-1.  1. -1.  1.]


In [4]:
m = load_model("pinch_full")

In [15]:
mujoco.mjs_findElement(spec, mujoco.mjtObj.mjOBJ_FRAME, "rh_frame")


AttributeError: module 'mujoco' has no attribute 'mjs_findElement'

In [7]:
xml_string = mujoco.mj_saveXML(m, None)

# Parse XML string into MjSpec
spec = mujoco.MjSpec.from_string(xml_string)

AttributeError: module 'mujoco' has no attribute 'mj_saveXML'

In [21]:
f = 'scene_mjx_cube_pinch.xml'
m = mujoco.MjModel.from_xml_path('../mujoco_playground/mujoco_playground/_src/manipulation/tesollo_hand/xmls/'+f)
data = mujoco.MjData(m)

In [ ]:
v = mujoco.viewer.launch_passive(m, data)
if v.is_running():                                                                                                                                                                               
    mujoco.mj_step(m, data)
    v.sync()                                                                                                                                                                   

In [ ]:
f = 'tesollo_wrist_dof_no_actuators.xml'

m = mujoco.MjModel.from_xml_path('../mujoco_playground/mujoco_playground/_src/manipulation/tesollo_hand/xmls/'+f)
data = mujoco.MjData(m)

# Load the home keyframe as starting pose
mujoco.mj_resetDataKeyframe(m, data, 0)

def print_qpos(keycode):
    if chr(keycode) == 'P':
        jnt_names = [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(m.njnt)]
        print('\n--- current qpos ---')
        print(' '.join(f'{v:.6f}' for v in data.qpos))
        print('--- per joint ---')
        qi = 0
        for i, name in enumerate(jnt_names):
            jtype = m.jnt_type[i]
            nq = 4 if jtype == mujoco.mjtJoint.mjJNT_FREE else (
                 3 if jtype == mujoco.mjtJoint.mjJNT_BALL else 1)
            vals = data.qpos[qi:qi+nq]
            print(f'  {name}: {" ".join(f"{v:.6f}" for v in vals)}')
            qi += nq
        print()

v = mujoco.viewer.launch_passive(m, data)


In [30]:
m.body("rh").quat = np.array([1, 0, 1, 0])
mujoco.mj_forward(m, data)

In [5]:
# Test: load scene_mjx_cube_wrist_dof.xml and inspect any error
for f in ['reorientation_cube.xml', 'scene_mjx_cube.xml','scene_mjx_cube_wrist_dof.xml']:                                                                                                                   
    try:   
        m = mujoco.MjModel.from_xml_path('../mujoco_playground/mujoco_playground/_src/manipulation/tesollo_hand/xmls/'+f)
        print(f, 'OK nbody=', m.nbody, 'nmesh=', m.nmesh)
    except Exception as e:
        print(f, 'ERR', str(e)[:200])

reorientation_cube.xml OK nbody= 2 nmesh= 1
scene_mjx_cube.xml ERR Error: Error opening file 'meshes/mujoco_playground/mujoco_playground/_src/manipulation/tesollo_hand/xmls/meshes/dex_cube.obj'
scene_mjx_cube_wrist_dof.xml ERR Error: Error opening file 'meshes/mujoco_playground/mujoco_playground/_src/manipulation/tesollo_hand/xmls/meshes/dex_cube.obj'


In [ ]:

                                                                                                                                                                                                        
# Build the env                                                                                                                                                                                         
env = pinch.CubePinch()                                                                                                                                                                           
mj_model = env.mj_model                                                                                                                                                                                 
                                                                                                                                                                                                        
# Load model into C MuJoCo and reset to keyframe                                                                                                                                                        
mujoco.mj_resetDataKeyframe(mj_model, data, 0)                                                                                                                                                          
                                                                                                                                                                                                        
# Open the passive (non-blocking) viewer
with mujoco.viewer.launch_passive(mj_model, data) as v:                                                                                                                                                 
    while v.is_running():                                                                                                                                                                               
        mujoco.mj_step(mj_model, data)
        v.sync()